In [22]:
import pandas as pd
import joblib

# Load model
model = joblib.load(
    "../data/processed/logistic_regression_model.pkl"
)

# Load scaler
scaler = joblib.load(
    "../data/processed/scaler.pkl"
)

# Load threshold
threshold = joblib.load(
    "../data/processed/churn_threshold.pkl"
)

print("Model loaded:", type(model).__name__)
print("Scaler loaded:", type(scaler).__name__)
print("Threshold:", threshold)

Model loaded: LogisticRegression
Scaler loaded: StandardScaler
Threshold: 0.4


In [9]:
print("Number of features:", len(model.feature_names_in_))
print("\nFeatures:")
print(model.feature_names_in_)

Number of features: 30

Features:
['SeniorCitizen' 'tenure' 'MonthlyCharges' 'TotalCharges' 'gender_Male'
 'Partner_Yes' 'Dependents_Yes' 'PhoneService_Yes'
 'MultipleLines_No phone service' 'MultipleLines_Yes'
 'InternetService_Fiber optic' 'InternetService_No'
 'OnlineSecurity_No internet service' 'OnlineSecurity_Yes'
 'OnlineBackup_No internet service' 'OnlineBackup_Yes'
 'DeviceProtection_No internet service' 'DeviceProtection_Yes'
 'TechSupport_No internet service' 'TechSupport_Yes'
 'StreamingTV_No internet service' 'StreamingTV_Yes'
 'StreamingMovies_No internet service' 'StreamingMovies_Yes'
 'Contract_One year' 'Contract_Two year' 'PaperlessBilling_Yes'
 'PaymentMethod_Credit card (automatic)' 'PaymentMethod_Electronic check'
 'PaymentMethod_Mailed check']


In [23]:
new_customer = {
    "SeniorCitizen": 0,
    "tenure": 12,
    "MonthlyCharges": 70.50,
    "TotalCharges": 846.00,
    "gender": "Male",
    "Partner": "No",
    "Dependents": "No",
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check"
}

customer_df = pd.DataFrame([new_customer])

customer_df

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod
0,0,12,70.5,846.0,Male,No,No,Yes,No,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check


In [ ]:
categorical_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

customer_encoded = pd.get_dummies(
    customer_df,
    columns=categorical_cols,
    drop_first=True
)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
0,0,12,70.5,846.0


In [24]:
customer_encoded = customer_encoded.reindex(
    columns=model.feature_names_in_,
    fill_value=0
)

print("Shape:", customer_encoded.shape)
print("Features match:",
      list(customer_encoded.columns) == list(model.feature_names_in_))

Shape: (1, 30)
Features match: True


In [25]:
num_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

customer_encoded[num_cols] = scaler.transform(
    customer_encoded[num_cols]
)

print(customer_encoded[num_cols])

   SeniorCitizen    tenure  MonthlyCharges  TotalCharges
0      -0.439319 -0.837902        0.182711     -0.639822


In [26]:
churn_probability = model.predict_proba(
    customer_encoded
)[0, 1]

prediction = int(churn_probability >= threshold)

print(f"Churn Probability: {churn_probability:.2%}")

if prediction == 1:
    print("Prediction: Customer is likely to churn")
else:
    print("Prediction: Customer is likely to stay")

Churn Probability: 30.71%
Prediction: Customer is likely to stay


In [33]:
def predict_churn(customer_data):
    
    # Convert input to DataFrame
    customer_df = pd.DataFrame([customer_data])

    # Categorical columns
    categorical_cols = [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod"
    ]

    # One-hot encoding
    customer_encoded = pd.get_dummies(
        customer_df,
        columns=categorical_cols,
        drop_first=True
    )

    # Match model features
    customer_encoded = customer_encoded.reindex(
        columns=model.feature_names_in_,
        fill_value=0
    )

    # Numerical columns
    num_cols = [
        "SeniorCitizen",
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ]

    # Scale numerical features
    customer_encoded[num_cols] = scaler.transform(
        customer_encoded[num_cols]
    )

    # Churn probability
    churn_probability = model.predict_proba(
        customer_encoded
    )[0, 1]

    # Apply threshold
    prediction = int(churn_probability >= threshold)

    # Return result
    return {
    "churn_probability": round(float(churn_probability), 4),
    "prediction": prediction,
    "result": "Likely to Churn" if prediction == 1 else "Likely to Stay"
}

In [34]:
result = predict_churn(new_customer)

print(result)

{'churn_probability': 0.3071, 'prediction': 0, 'result': 'Likely to Stay'}


In [32]:
#the second customer 
high_risk_customer = {
    "SeniorCitizen": 1,
    "tenure": 2,
    "MonthlyCharges": 95.00,
    "TotalCharges": 190.00,
    "gender": "Female",
    "Partner": "No",
    "Dependents": "No",
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check"
}

result = predict_churn(high_risk_customer)

print(result)


{'churn_probability': np.float64(0.3533315444995003), 'prediction': 0, 'result': 'Likely to Stay'}


In [35]:
very_high_risk_customer = {
    "SeniorCitizen": 1,
    "tenure": 1,
    "MonthlyCharges": 95.00,
    "TotalCharges": 95.00,
    "gender": "Female",
    "Partner": "No",
    "Dependents": "No",
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check"
}

result = predict_churn(very_high_risk_customer)

print(result)

{'churn_probability': 0.3598, 'prediction': 0, 'result': 'Likely to Stay'}


In [37]:
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(X_test.shape)
print(y_test.shape)

(1407, 30)
(1407,)


In [38]:
test_probabilities = model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= threshold
).astype(int)

print("First 10 probabilities:")
print(test_probabilities[:10])

print("\nFirst 10 predictions:")
print(test_predictions[:10])

print("\nFirst 10 actual values:")
print(y_test.iloc[:10].values)

First 10 probabilities:
[0.01802218 0.59183207 0.00489153 0.20142434 0.10376232 0.47052094
 0.02638963 0.16510364 0.67642949 0.01571737]

First 10 predictions:
[0 1 0 0 0 1 0 0 1 0]

First 10 actual values:
[0 0 0 1 0 1 0 0 1 0]
